In [61]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [73]:
base_path = 'data'
folders = ['idle', 'running', 'walking', 'stairs']

all_features = []
labels = []

for folder in folders:
    folder_path = os.path.join(base_path, folder)

    for file_name in os.listdir(folder_path):
        if file_name.endswith('.csv'):
            file_path = os.path.join(folder_path, file_name)

            df = pd.read_csv(file_path)

            features = {}
            features['X_mean'] = df['accelerometer_X'].mean()
            features['X_std'] = df['accelerometer_X'].std()
            features['Y_mean'] = df['accelerometer_Y'].mean()
            features['Y_std'] = df['accelerometer_Y'].std()
            features['Z_mean'] = df['accelerometer_Z'].mean()
            features['Z_std'] = df['accelerometer_Z'].std()
            features['X_max'] = df['accelerometer_X'].max()
            features['X_min'] = df['accelerometer_X'].min()
            features['Y_max'] = df['accelerometer_Y'].max()
            features['Y_min'] = df['accelerometer_Y'].min()
            features['Z_max'] = df['accelerometer_Z'].max()
            features['Z_min'] = df['accelerometer_Z'].min()

            all_features.append(features)
            labels.append(folder)

X = pd.DataFrame(all_features)
y = pd.Series(labels)

In [74]:
print(y.value_counts())

running    3408
walking    1850
idle       1039
stairs      165
Name: count, dtype: int64


In [75]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [76]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [77]:
svm_model = SVC(kernel='rbf', C=1.0, random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_predictions = svm_model.predict(X_test_scaled)

In [78]:
print(classification_report(y_test, svm_predictions))

              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       208
     running       1.00      1.00      1.00       682
      stairs       0.91      0.61      0.73        33
     walking       0.97      0.99      0.98       370

    accuracy                           0.99      1293
   macro avg       0.97      0.90      0.93      1293
weighted avg       0.99      0.99      0.99      1293



In [79]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_predictions = rf_model.predict(X_test_scaled)

In [80]:
print(classification_report(y_test, rf_predictions))

              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       208
     running       1.00      1.00      1.00       682
      stairs       1.00      0.91      0.95        33
     walking       0.99      1.00      1.00       370

    accuracy                           1.00      1293
   macro avg       1.00      0.98      0.99      1293
weighted avg       1.00      1.00      1.00      1293



Random Forest показав себе значно краще за SVM. Оскільки датасет сильно незбалансований (для running понад 3400 записів, а для stairs лише 165), SVM дуже погано розпізнає найменший клас.

Також додавання нових фічів (мінімуму та максимуму до вже наявних середнього та стандартного відхилення) покращило результати обох алгоритмів. Для класу stairs у SVM f1-score зріс з 0.68 до 0.73, а у Random Forest — з 0.89 до 0.95

Загалом найкраще підійшла модель Random Forest на розширеному наборі фічів. Додаткові часові ознаки реально допомогли моделям краще розпізнавати сходинки.